In [21]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama2")

In [22]:
llm.invoke("how can langsmith help with testing?")

"\nLangsmith is a language model that can assist with testing in several ways:\n\n1. Automated Testing: Langsmith can be used to generate automated tests for software applications, helping to ensure that the application behaves as expected and detecting any bugs or issues before they become problems.\n2. Code Review: Langsmith can be used to review code and identify potential issues or vulnerabilities, such as syntax errors, logical inconsistencies, and security risks.\n3. Test Data Generation: Langsmith can generate test data that is tailored to a specific application or use case, helping to ensure that the tests are comprehensive and cover all possible scenarios.\n4. Regression Testing: Langsmith can be used to generate regression tests, which are designed to verify that changes made to an application have not introduced new bugs or issues.\n5. Automated Test Maintenance: Langsmith can be used to maintain automated tests over time, ensuring that they remain relevant and effective as 

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a world class technical documentation writer."),
        ("user", "{input}"),
    ]
)

In [24]:
chain = prompt | llm

In [25]:
chain.invoke({"input": "how can langsmith help with testing?"})

"\nAs a world-class technical documentation writer, I must say that LingSMITH is an incredible tool for testing and ensuring the quality of your technical content. Here are some ways in which LingSMITH can help with testing:\n\n1. Grammar and Spell Checking: LingSMITH's advanced grammar and spell checking capabilities can help identify and correct even the most subtle errors in your technical documentation. This includes contextual spelling suggestions, grammatical corrections, and more.\n2. Sentence Structure Analysis: LingSMITH can analyze the structure of your sentences and identify any inconsistencies or awkward phrasing. This can help improve the readability and clarity of your technical content.\n3. Syntax Analysis: LingSMITH can analyze the syntax of your code and identify potential errors, inconsistencies, or areas for improvement. This can help ensure that your code is readable, maintainable, and efficient.\n4. Style Guidance: LingSMITH can provide guidance on coding styles an

In [26]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [27]:
chain = prompt | llm | output_parser

In [28]:
chain.invoke({"input": "how can langsmith help with testing?"})

"\nAh, an excellent question! As a world-class technical documentation writer, I must say that Langsmith is an incredibly versatile tool that can greatly aid in the testing process. Here are some ways Langsmith can help:\n\n1. Automated Testing: Langsmith's natural language processing (NLP) capabilities allow it to generate test cases from your technical documentation, saving you time and effort. This feature ensures that your tests are comprehensive and cover all aspects of your product or service.\n2. Documentation Integration: Langsmith can integrate with your existing documentation systems, such as Doxygen, Read the Docs, or Sphinx. By doing so, it enables you to generate test cases from your documentation, ensuring that your tests are aligned with your technical writing.\n3. Test Data Generation: Langsmith's machine learning algorithms can generate test data based on your product's features and functionality. This feature is particularly useful when testing complex systems or APIs

In [29]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")

docs = loader.load()

In [30]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings()

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

In [32]:
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:

    <context>
    {context}
    </context>

    Question: {input}
"""
)

document_chain = create_stuff_documents_chain(llm, prompt)

In [33]:
from langchain_core.documents import Document

document_chain.invoke(
    {
        "input": "how can langsmith help with testing?",
        "context": [
            Document(page_content="langsmith can let you visualize test results")
        ],
    }
)

'Based on the provided context, Langsmith can help with testing by allowing users to visualize their test results.'

In [34]:
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retriever_chain = create_retrieval_chain(retriever, document_chain)

In [35]:
response = retriever_chain.invoke({"input": "how can langsmith help with testing?"})
print(response["answer"])

# LangSmith offers several features that can help with testing:...

 Based on the provided context, LangSmith can help with testing in several ways:

1. Creating test cases: LangSmith allows developers to create datasets, which are collections of inputs and reference outputs, and use these to run tests on their LLM applications. Developers can upload these test cases in bulk, create them on the fly, or export them from application traces.
2. Annotating traces: LangSmith supports sending runs to annotation queues, allowing annotators to closely inspect interesting traces and annotate them with respect to different criteria. This helps users catch regressions across important evaluation criteria.
3. Adding runs to a dataset: As the application progresses through the beta testing phase, it's essential to continue collecting data to refine and improve its performance. LangSmith enables users to add runs as examples to datasets, expanding test coverage on real-world scenarios.
4. Monitoring and A/B testing: LangSmith provides monitoring charts that allow us

In [36]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# First we need a prompt that we can pass into an LLM to generate this search query

prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        (
            "user",
            "Given the above conversation, generate a search query to look up to get information relevant to the conversation",
        ),
    ]
)
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

In [37]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="Can LangSmith help test my LLM applications?"),
    AIMessage(content="Yes!"),
]
retriever_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "Tell me how",
    }
)

[Document(metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'LangSmith User Guide | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.', 'language': 'en'}, page_content="Every playground run is logged in the system and can be used to create test cases or compare with other runs.Beta Testing\u200bBeta testing allows developers to collect more data on how their LLM applications are performing in real-world scenarios. In this phase, it’s important to develop an understanding for the types of inputs the app is performing well or poorly on and how exactly it’s breaking down in those cases. Both feedback colle

In [38]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user's questions based on the below context:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
    ]
)
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

In [39]:
chat_history = [
    HumanMessage(content="Can LangSmith help test my LLM applications?"),
    AIMessage(content="Yes!"),
]
retrieval_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "Tell me how",
    }
)

{'chat_history': [HumanMessage(content='Can LangSmith help test my LLM applications?'),
  AIMessage(content='Yes!')],
 'input': 'Tell me how',
 'context': [Document(metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'LangSmith User Guide | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.', 'language': 'en'}, page_content="Every playground run is logged in the system and can be used to create test cases or compare with other runs.Beta Testing\u200bBeta testing allows developers to collect more data on how their LLM applications are performing in real-world scenarios. In this phase, it’s important to deve